In [ ]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [ ]:
chandan = fetch_website_contents("https://chandankumarshani.vercel.app/")
print(chandan)

## Types of prompts


Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [ ]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [ ]:
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```

In [ ]:
# Creating a function to generate the formatted messages for the LLM

def messages_for(content):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + content}
    ]

In [ ]:
messages_for(chandan)

In [ ]:
load_dotenv(override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")

if not gemini_api_key:
    print("No Gemini API key was found.")
elif not gemini_api_key.startswith("AQ."):
    print("A Gemini API key was found, but it doesn't start AQ.; please check you're using the right key.")
else:
    print("Gemini API key found and looks good so far!")

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=gemini_api_key)

In [ ]:
# function to that combines everything together and generates the summary for a given website

def summarize(url):
    content = fetch_website_contents(url)
    response = gemini.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages = messages_for(content)
    )
    return response.choices[0].message.content

In [ ]:
summarize("https://chandankumarshani.vercel.app/")

In [ ]:
# Function to display the summary nicely in markdown format

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://chandankumarshani.vercel.app/")

In [ ]:
display_summary("https://www.starbucks.de/")

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
message = "Hello, LLama! I am chandan and I am testing you out. Can you tell me a joke?"

response = ollama.chat.completions.create(
    model="llama3.2",
    messages = [{"role": "user", "content": message}]
)

response.choices[0].message.content

## Using Open source Llama model to summarize website

In [ ]:
def summarize_open(url):
    content = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages = messages_for(content)
    )
    return response.choices[0].message.content

In [ ]:
def display_summary_v2(url):
    summary = summarize_open(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://ollama.com/")